# 부록 A. Colab에서 Git 기초 · Appendix A — Git Basics in Colab

전산실험 2 · AI Agent 만들기 · **2주차** · 예상 소요 30분
Computational Experiments 2 · Building AI Agents · **Week 2** · about 30 minutes

## 목표 · Objectives
- `init` / `status` / `add` / `commit` / `log` / `diff` 를 직접 사용한다
- **작업 디렉터리 → 스테이징 → 저장소** 세 공간을 구분한다
- 과거 버전으로 되돌린다
- **프롬프트도 버전 관리 대상**이라는 것을 체감한다

> **EN**
> - Use `init` / `status` / `add` / `commit` / `log` / `diff` yourself
> - Tell the three spaces apart: **working directory → staging area → repository**
> - Roll back to an earlier version
> - Feel for yourself that **prompts belong under version control too**

> Colab은 리눅스 컴퓨터입니다. 셀 앞에 `!` 를 붙이면 셸 명령이 실행됩니다.
> **EN** — Colab is a Linux machine; prefix a line with `!` to run a shell command.

> **이 노트북은 2주차 부록입니다.** 본 진도(예제 01~03: 토큰·컨텍스트·비용)와
> 별개로, 앞으로 12주 동안 쓸 작업 환경을 갖추는 인프라 실습입니다.
> 오늘 못 끝내도 됩니다. 다만 **4주차 전에는 반드시 마쳐 두십시오.**
>
> **EN — this notebook is a week-2 appendix.** Separate from the main track
> (Examples 01–03: tokens, context, cost), it sets up the working environment you
> will use for the next twelve weeks. You need not finish it today, but
> **you must finish it before week 4.**

---
## 0. 왜 이걸 배우는가 · Why learn this

지난주에 우리는 같은 함수를 노트북 네 개에 복사해 붙였습니다.

에이전트를 만들다 보면 여기에 하나가 더 붙습니다. **프롬프트를 수십 번 고치게 됩니다.**
그러다 "어제 버전이 더 잘 됐던 것 같은데" 하는 순간이 반드시 옵니다.
그때 기록이 없으면 방법이 없습니다.

**프롬프트도 코드입니다.**

> **EN** — Last week we copy-pasted the same functions into four notebooks. As you
> build an agent, one more thing joins them: **you will revise your prompts dozens
> of times.** Sooner or later comes the moment of "I think yesterday's version was
> better" — and with no record, there is nothing you can do.
>
> **A prompt is code.**

In [ ]:
# Git 은 커밋에 '누가 했는지'를 기록합니다. 먼저 이름과 이메일을 설정합니다.
# GitHub 계정과 같은 이메일을 쓰면 커밋이 프로필에 연결됩니다.
# EN — Git records WHO made each commit, so set your name and email first.
#      Using the same email as your GitHub account links commits to your profile.

!git config --global user.name  "본인이름"
!git config --global user.email "본인이메일@example.com"

!git config --global --list

---
## 1. 저장소 만들기 · Creating a repository

`git init` 은 "이 폴더의 변경 이력을 관리하겠다"고 선언하는 명령입니다.
숨김 폴더 `.git` 이 생기고, 여기에 모든 이력이 저장됩니다.

> **EN** — `git init` declares "I will track the history of this folder". It
> creates the hidden `.git` directory, where the entire history lives.

In [ ]:
# 실습용 폴더를 만들고 그 안으로 이동
# 주의: !cd 는 셀이 끝나면 원위치합니다. 디렉터리 이동은 %cd 를 써야 합니다.
# EN — Create a practice folder and move into it. Note that `!cd` reverts when the
#      cell ends; use `%cd` to actually change directory.
!rm -rf /content/prompt_lab
!mkdir -p /content/prompt_lab
%cd /content/prompt_lab

!git init
!ls -a

---
## 2. 세 공간 · The three spaces

```
작업 디렉터리        스테이징 영역         저장소
working directory    staging area        repository
 (수정 중)     --->   (커밋 예정)   --->    (기록됨)
 (being edited)      (staged)             (recorded)
              add                  commit
```

파일을 만들어도 Git은 아직 아무것도 기록하지 않습니다.
`git status` 로 지금 어느 단계인지 항상 확인하세요.

> **EN** — Creating a file records nothing in Git yet. Use `git status` constantly
> to see which stage you are at.

In [ ]:
# 1주차에 썼던 프롬프트를 파일로 저장 / Save the week-1 prompt as a file
prompt_v1 = "1부터 100 사이의 정수를 하나만 고르세요. 숫자만 답하세요."

with open("guess_prompt.txt", "w", encoding="utf-8") as f:
    f.write(prompt_v1)

!cat guess_prompt.txt

In [ ]:
# Git 이 보기에 지금 상태는? / What does Git think the state is?
!git status

`Untracked files` 라고 나옵니다.
"이런 파일이 있는데 나는 아직 관리하지 않는다"는 뜻입니다.

> **EN** — It says `Untracked files`, meaning "there is a file here, but I am not
> managing it yet".

In [ ]:
# 스테이징: "이번 커밋에 포함하겠다"고 표시
# EN — Staging: mark it as "include this in the next commit"
!git add guess_prompt.txt
!git status

In [ ]:
# 커밋: 확정해서 이력에 남긴다 / Commit: fix it into the history
!git commit -m "숫자 맞히기 프롬프트 최초 작성"
!git log --oneline

### 커밋 메시지 쓰는 법 · How to write a commit message

| 나쁜 예 | 좋은 예 |
|---|---|
| `수정` | `재시도 로직 추가` |
| `ㅇㅇ` | `프롬프트에 이분탐색 전략 지시 추가` |
| `최종` | `파싱 실패 시 재요청하도록 변경` |

**EN**

| Bad | Good |
|---|---|
| `fix` | `add retry logic` |
| `wip` | `instruct the prompt to use binary search` |
| `final` | `re-request on a parse failure` |

**무엇을 왜 했는지** 쓰십시오. 석 달 뒤의 자신은 완전히 남입니다.

> **EN** — Write **what you did and why.** Three months from now, you are a
> stranger to yourself.

---
## 3. 고치고, 무엇이 바뀌었는지 보기 · Edit, then see what changed

`git diff` 는 Git의 진짜 힘입니다.
파일 두 개를 눈으로 비교할 필요가 없습니다.

> **EN** — `git diff` is where Git's real power lies: you never have to compare two
> files by eye.

In [ ]:
# 프롬프트 개선 (1주차 연습문제에서 나온 아이디어)
# EN — Improve the prompt, using the idea from the week-1 exercise
prompt_v2 = (
    "1부터 100 사이의 정수를 하나만 고르세요.\n"
    "이분탐색 전략을 사용하세요: 가능한 범위의 중간값을 시도하세요.\n"
    "반드시 숫자만 답하고 다른 말은 하지 마세요."
)

with open("guess_prompt.txt", "w", encoding="utf-8") as f:
    f.write(prompt_v2)

# 무엇이 바뀌었나? / What changed?
!git diff

`-` 로 시작하는 줄이 지워진 것, `+` 로 시작하는 줄이 추가된 것입니다.

> **EN** — Lines beginning with `-` were removed; lines beginning with `+` were
> added.

In [ ]:
!git add guess_prompt.txt
!git commit -m "프롬프트에 이분탐색 전략 지시와 형식 제약 추가"

# 프롬프트를 한 번 더 개선 -- 출력 예시를 준다 (few-shot)
# EN — Improve the prompt once more, giving an output example (few-shot)
prompt_v3 = prompt_v2 + "\n\n예시 답변: 50"

with open("guess_prompt.txt", "w", encoding="utf-8") as f:
    f.write(prompt_v3)

!git add guess_prompt.txt
!git commit -m "프롬프트에 출력 예시(few-shot) 추가"

!git log --oneline

---
## 4. 이력 살펴보기와 되돌리기 · Reading the history and rolling back

세 개의 커밋이 쌓였습니다. 이제 시간 여행이 가능합니다.

> **EN** — Three commits have accumulated, and time travel is now available.

In [ ]:
# 커밋 목록을 자세히 / The commit list, in more detail
!git log --pretty=format:"%h  %ad  %s" --date=short

In [ ]:
# 첫 커밋과 현재의 차이 / The difference between the first commit and now
!git diff HEAD~2 HEAD -- guess_prompt.txt

In [ ]:
# 특정 시점의 파일 내용 보기 (되돌리지 않고 구경만)
# EN — View the file as of a given commit, without rolling anything back
print("=== 최초 버전 ===")
!git show HEAD~2:guess_prompt.txt
print()
print("=== 현재 버전 ===")
!cat guess_prompt.txt

In [ ]:
# 실제로 되돌리기: 파일을 최초 버전으로
# EN — Actually roll back: restore the file to its first version
!git checkout HEAD~2 -- guess_prompt.txt
!cat guess_prompt.txt
print()
!git status

In [ ]:
# 되돌린 것을 취소하고 최신 버전으로 복귀
# EN — Undo the rollback and return to the latest version
!git checkout HEAD -- guess_prompt.txt
!cat guess_prompt.txt

> **`git checkout -- <파일>` 은 저장하지 않은 수정을 지웁니다.**
> 되돌리기 전에 `git status` 로 확인하는 습관을 들이세요.

> **EN — `git checkout -- <file>` discards uncommitted edits.** Get into the habit
> of checking `git status` before you roll anything back.

---
## 5. `.gitignore` — 올리지 말아야 할 것 · What must never be committed

모든 파일을 관리할 필요는 없습니다. 특히 **비밀정보**는 절대 안 됩니다.

> **EN** — Not every file needs tracking, and **secrets** must never be tracked at
> all.

In [ ]:
gitignore = """\
# 비밀정보 - 절대 커밋 금지
.env
*.key

# 파이썬
__pycache__/
.ipynb_checkpoints/

# 큰 데이터
*.csv
"""

with open(".gitignore", "w", encoding="utf-8") as f:
    f.write(gitignore)

# 무시되는지 확인용 파일들을 만들어 본다
# EN — Create some files to confirm they really are ignored
with open(".env", "w") as f:
    f.write("KMU_API_KEY=이건_비밀입니다")
with open("data.csv", "w") as f:
    f.write("a,b\n1,2")

!git status --short

`.env` 와 `data.csv` 가 목록에 **나타나지 않습니다.** 의도한 대로입니다.

> **EN** — `.env` and `data.csv` **do not appear** in the list. Exactly as intended.

In [ ]:
!git add .gitignore
!git commit -m ".gitignore 추가 - 비밀정보와 데이터 제외"
!git log --oneline

---
## 연습문제 · Exercises

**문제 1.** `guess_prompt.txt` 를 두 번 더 수정하며 커밋하시오.
그 다음 `git log --oneline` 으로 이력을 확인하고,
가장 오래된 커밋과 최신 커밋의 차이를 `git diff` 로 출력하시오.

**문제 2.** `prompts/` 폴더를 만들고 1주차에 썼던 프롬프트들
(system 메시지, 형식 지시 프롬프트 등)을 각각 파일로 저장한 뒤 커밋하시오.

**문제 3.** `git commit --amend -m "새 메시지"` 는 무엇을 하는가?
직접 실행해서 `git log` 가 어떻게 달라지는지 확인하시오.
이미 push 한 커밋에 이것을 쓰면 왜 곤란한가?

**문제 4.** `.gitignore` 에 `*.csv` 를 넣기 **전에** 이미 커밋된 csv 파일이 있다면
어떻게 되는가? 직접 실험해 보시오.

> **EN — 1.** Edit and commit `guess_prompt.txt` twice more, review the history
> with `git log --oneline`, and print the difference between the oldest and the
> newest commit with `git diff`.
>
> **EN — 2.** Create a `prompts/` folder, save each of the week-1 prompts (the
> system messages, the format-instruction prompt and so on) as its own file, and
> commit them.
>
> **EN — 3.** What does `git commit --amend -m "new message"` do? Run it and see
> how `git log` changes. Why is it a problem to use it on a commit you have
> already pushed?
>
> **EN — 4.** What happens to a csv file that was already committed **before** you
> added `*.csv` to `.gitignore`? Try it and find out.

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary><b>해설 보기 · Show the solutions</b></summary>

**문제 3.**
`--amend` 는 **직전 커밋을 새 커밋으로 바꿔치기** 합니다.
커밋 해시가 달라지므로, 이미 push 한 커밋을 amend 하면 원격의 이력과 어긋나
`push` 가 거부됩니다. 강제 push(`--force`)로 밀어붙이면 같은 저장소를 쓰는
다른 사람의 이력이 깨집니다.
→ **혼자 쓰는 저장소에서, push 하기 전에만** 쓰십시오.

**문제 4.**
`.gitignore` 는 **아직 추적되지 않는 파일**에만 적용됩니다.
이미 커밋된 파일은 계속 추적됩니다. 빼려면 명시적으로 제거해야 합니다.
```
!git rm --cached data.csv
!git commit -m "추적 대상에서 data.csv 제외"
```
`--cached` 를 빼면 실제 파일까지 지워지니 주의하십시오.
그리고 **과거 커밋에는 여전히 남아 있습니다.** 이 사실이
`06_appendix_secrets_and_config.ipynb` 의 핵심입니다.

---

**EN — 3.** `--amend` **replaces the previous commit with a new one.** The commit
hash changes, so amending a commit you have already pushed puts you out of step
with the remote and `push` is refused. Forcing it through (`--force`) breaks the
history for everyone else sharing the repository. → Use it **only in a repository
you work in alone, and only before you push.**

**EN — 4.** `.gitignore` applies only to **files not yet tracked**. A file that
is already committed stays tracked; removing it takes an explicit step (see the
commands above). Leave off `--cached` and the real file is deleted too, so take
care. And **it remains in the past commits** — which is the whole point of
`06_appendix_secrets_and_config.ipynb`.

</details>

---
## 정리 · Summary

- `add`(스테이징) → `commit`(확정). `status` 로 항상 위치를 확인한다
- `diff` 로 무엇이 바뀌었는지 본다. 눈으로 비교하지 않는다
- 커밋 메시지에는 **무엇을 왜** 했는지 쓴다
- `.gitignore` 는 **처음부터** 만들어 둔다
- **프롬프트도 코드다**

> **EN**
> - `add` (stage) → `commit` (fix it down); use `status` to know where you are
> - Use `diff` to see what changed — never compare by eye
> - A commit message says **what and why**
> - Write `.gitignore` **from the start**
> - **A prompt is code**

**다음 노트북** → [`05_appendix_github_colab.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week02/05_appendix_github_colab.ipynb)